In [2]:
!pip install pyspark py4j

In [3]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 15.1 MB/s eta 0:00:00


In [4]:
import csv
from faker import Faker
import random

fake = Faker()

num_records = 100000

http_methods = ['GET', 'POST', 'PUT', 'DELETE']
response_codes = [200, 301, 404, 500]

file_path = "web_server_logs.csv"

with open(file_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['ip', 'timestamp', 'method', 'url', 'response_code', 'response_size'])

    for _ in range(num_records):
        ip = fake.ipv4()
        timestamp = fake.date_time_this_year().isoformat()
        method = random.choice(http_methods)
        url = fake.uri_path()
        response_code = random.choice(response_codes)
        response_size = random.randint(100, 10000)

        writer.writerow([ip, timestamp, method, url, response_code, response_size])

print(f"Сгенерировано {num_records} записей и сохранено в {file_path}")

Сгенерировано 100000 записей и сохранено в web_server_logs.csv


In [25]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, sum

spark = SparkSession.builder.appName("Read CSV Example").getOrCreate()

# Чтение CSV-файла
df = spark.read.csv("/content/web_server_logs.csv", header=True, inferSchema=True)

print('10 самых активных IP:')
df.groupBy(df.ip).count().withColumnRenamed('count', 'request_count').orderBy('request_count', ascending=False).limit(10).show()

print('Количество запросов по HTTP-методам:')
df.groupBy(df.method).count().withColumnRenamed('count', 'method_count').show()

print('Количество запросов кодом ответа 404:', df.filter(df.response_code == 404).count())

print('Сумма размера ответов по дням:')
cast_df = df.withColumns({
    'response_size': col('response_size').cast('int'),
    'date': to_date(col('timestamp'))
    })
cast_df.groupBy('date').agg(sum('response_size').alias('total_size')).orderBy('date').show()

10 самых активных IP:
+---------------+-------------+
|             ip|request_count|
+---------------+-------------+
| 21.189.148.252|            1|
|   13.94.173.17|            1|
|175.188.160.206|            1|
|   119.10.86.46|            1|
|  24.129.242.92|            1|
|  61.108.94.201|            1|
|  161.41.19.192|            1|
| 215.230.245.13|            1|
|194.241.225.248|            1|
|   77.172.6.174|            1|
+---------------+-------------+

Количество запросов по HTTP-методам:
+------+------------+
|method|method_count|
+------+------------+
|  POST|       25114|
|DELETE|       24889|
|   PUT|       25011|
|   GET|       24986|
+------+------------+

Количество запросов кодом ответа 404: 24842
Сумма размера ответов по дням:
+----------+----------+
|      date|total_size|
+----------+----------+
|2026-01-01|   3041880|
|2026-01-02|   3247807|
|2026-01-03|   2903616|
|2026-01-04|   3075048|
|2026-01-05|   3028732|
|2026-01-06|   3063030|
|2026-01-07|   3018517|
